# 01 — EDA REES46 (sample)
Khám phá nhanh trên `data/sample/rees46_sample.csv.gz` (tạo bằng `make sample`).
Mục tiêu: hiểu phân phối event_type, tỷ lệ null của `category_code`/`brand`, độ dài session, giá — để chốt DQ rules và feature.

In [ ]:
import sys; sys.path.insert(0, '/opt/app/libs')
from ecom_common.spark import get_spark
from pyspark.sql import functions as F
spark = get_spark('eda', {'spark.master': 'local[*]'})
df = (spark.read.option('header', True).option('inferSchema', True)
      .csv('/data/sample/rees46_sample.csv.gz'))
df.printSchema(); df.count()

In [ ]:
df.groupBy('event_type').count().orderBy(F.desc('count')).show()

In [ ]:
df.select([F.round(F.avg(F.col(c).isNull().cast('int')), 4).alias(c) for c in df.columns]).show(vertical=True)

In [ ]:
sess = df.groupBy('user_session').agg(F.count('*').alias('n'), F.max(F.col('event_type') == 'purchase').alias('converted'))
sess.select(F.expr('percentile_approx(n, array(0.5, 0.9, 0.99))').alias('p50_p90_p99'), F.avg(F.col('converted').cast('int')).alias('cvr')).show(truncate=False)

In [ ]:
df.select(F.min('price'), F.expr('percentile_approx(price, 0.5)'), F.max('price'), F.sum((F.col('price') <= 0).cast('int')).alias('non_positive')).show()

In [ ]:
(df.withColumn('l1', F.split('category_code', r'\\.')[0]).groupBy('l1')
   .agg(F.count('*').alias('events'), F.sum((F.col('event_type')=='purchase').cast('int')).alias('purchases'))
   .orderBy(F.desc('events')).show(20))